# BERTopic: Topic Modelling on 10-K MD&A Sentences

**Pipeline**

1. GPU setup
2. Install dependencies and verify environment compatibility
3. Load sentence-level data and prepare chunk-level corpus
4. Fit an initial BERTopic model and inspect topic visuals
5. Validate coherence and compare clustering strategies
6. Fit final model with FinBERT embeddings + KeyBERTInspired representation
7. Export topic outputs for downstream web app integration


## 0. Setup


In [1]:
# ── cuML (RAPIDS) GPU acceleration ───────────────────────────────────────────
# We detect cuml via importlib — no actual import here — so cuml.accel does
# NOT auto-activate before hdbscan is imported in the next cell.
# cuML UMAP runs on GPU; hdbscan is imported first (cell 5) so CpuHDBSCAN is
# bound to the real CPU class before cuml can proxy it.

import subprocess, sys, importlib.util

def _cuml_available():
    return importlib.util.find_spec('cuml') is not None

if not _cuml_available():
    print('Installing cuml-cu12 …')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--extra-index-url', 'https://pypi.nvidia.com', 'cuml-cu12'],
        capture_output=True, text=True
    )

USE_CUML = _cuml_available()
print('✅ cuML found — UMAP on GPU, HDBSCAN on CPU (imported before cuml).' if USE_CUML
      else 'ℹ️  cuML not found — UMAP & HDBSCAN on CPU.')


✅ cuML found — UMAP on GPU, HDBSCAN on CPU (imported before cuml).


In [2]:
import torch

assert torch.cuda.is_available(), "CUDA not available — check cluster GPU allocation."
DEVICE = "cuda"

print(f"✅ Using device : {DEVICE}")
print(f"   GPU count    : {torch.cuda.device_count()}")
print(f"   GPU name     : {torch.cuda.get_device_name(0)}")
print(
    f"   VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
)

✅ Using device : cuda
   GPU count    : 1
   GPU name     : NVIDIA GeForce RTX 3090
   VRAM         : 25.3 GB


In [3]:
# GPU diagnostics (NVIDIA only)
import subprocess, shutil

if DEVICE == "cuda" and shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout)


Sat Apr  4 22:04:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.44.01              Driver Version: 590.44.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:81:00.0 Off |                  N/A |
| 30%   42C    P3            122W /  350W |       4MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install -q "numpy<2.3" "numba>=0.60,<0.62" "umap-learn>=0.5.7" "hdbscan>=0.8.33" bertopic sentence-transformers gensim

import warnings, random, time
import numpy as np

assert np.lib.NumpyVersion(np.__version__) < np.lib.NumpyVersion("2.3.0"), (
    "NumPy >=2.3 detected — restart kernel and re-run this cell."
)

warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from IPython.display import display

# Import hdbscan BEFORE cuml — ensures CpuHDBSCAN is the real CPU class
# even if cuml.accel auto-activates and patches hdbscan.HDBSCAN afterwards.
from hdbscan import HDBSCAN as CpuHDBSCAN
from umap import UMAP as CpuUMAP
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator

# Import cuML UMAP *after* hdbscan so the proxy only affects the module
# attribute, not our already-bound CpuHDBSCAN reference.
if USE_CUML:
    from cuml.manifold import UMAP as GpuUMAP
else:
    GpuUMAP = CpuUMAP
GpuHDBSCAN = CpuHDBSCAN  # always CPU — real class captured before cuml import

from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

plt.rcParams["figure.dpi"] = 110
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"✅ All imports OK — USE_CUML={USE_CUML}")

✅ All imports OK — USE_CUML=True


In [5]:
# ==================================================================
# RUN 2 HYPERPARAMETERS  (fixed - no sweep needed)
# ==================================================================
MIN_CLUSTER_SIZE = 50  # HDBSCAN - min sentences per topic
MIN_SAMPLES = 15  # HDBSCAN - controls cluster density
MIN_DF = 5  # CountVectorizer - min doc-frequency
NGRAM_RANGE = (1, 3)  # CountVectorizer - up to trigrams
BEST_MCS = MIN_CLUSTER_SIZE  # used by final model directly

print(f"Config  min_cluster_size={MIN_CLUSTER_SIZE}  min_samples={MIN_SAMPLES}")
print(f"        min_df={MIN_DF}  ngram_range={NGRAM_RANGE}")

Config  min_cluster_size=50  min_samples=15
        min_df=5  ngram_range=(1, 3)


## 1. Data Loading & Preparation

Load `mda_shared_preprocessed.csv` (sentence-level: 1 row per sentence) and aggregate
into filing-level documents (1 row per filing) for topic modeling.


In [6]:
# import os
from pathlib import Path

%pwd
DATA_PATH = Path("textmining_grp6/datasets/final/mda_shared_preprocessed.csv")
df = pd.read_csv(DATA_PATH)
print(f"cwd={Path.cwd()}")
print(f"loaded={len(df):,} rows")
display(df.head(5))

cwd=/common/home/users/w/wlcheng.2023
loaded=452,390 rows


,doc_id,company_name,filing_type,filing_date,year,quarter,sentence,sentiment
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...,neutral
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...,neutral
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...,neutral
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,we base our estimates and judgments on our his...,neutral
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,actual results may differ from these estimates...,neutral


In [7]:
# ── 3.1 Chunking parameters ───────────────────────────────────────────────────
CHUNK_SIZE = 5  # consecutive sentences per chunk (tune as needed)

# Assign stable sentence_id before any filtering.
# This index is shared with FinBERT: both models must see rows in the same order.
df["sentence_id"] = df.index

# ── 3.2 Create chunk_df + sent_chunk_map ─────────────────────────────────────
# sent_chunk_map is the critical bridge between:
#   FinBERT  (sentence-level output)  →  sentence_id
#   BERTopic (chunk-level input)      →  chunk_id
# Build it HERE — before either model runs — and save to CSV immediately.

chunk_rows = []
sent_chunk_rows = []
chunk_counter = 0

for (doc_id, company, year, quarter), grp in df.groupby(
    ["doc_id", "company_name", "year", "quarter"], sort=False
):
    sents = grp["sentence"].tolist()
    sent_ids = grp["sentence_id"].tolist()

    for start in range(0, len(sents), CHUNK_SIZE):
        chunk_sents = sents[start : start + CHUNK_SIZE]
        chunk_sent_ids = sent_ids[start : start + CHUNK_SIZE]
        cid = f"c{chunk_counter}"

        chunk_rows.append(
            {
                "chunk_id": cid,
                "doc_id": doc_id,
                "company": company,
                "quarter": quarter,
                "year": year,
                "chunk_text": " ".join(chunk_sents),
            }
        )
        for sid in chunk_sent_ids:
            sent_chunk_rows.append({"sentence_id": sid, "chunk_id": cid})

        chunk_counter += 1

chunk_df = pd.DataFrame(chunk_rows)
sent_chunk_map = pd.DataFrame(sent_chunk_rows)

# BERTopic input: one text string per chunk
docs = chunk_df["chunk_text"].tolist()

# Save sent_chunk_map immediately — needed by final_df in Section 14
OUT_DIR = Path("datasets/webapp_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
sent_chunk_map.to_csv(OUT_DIR / "sent_chunk_map.csv", index=False)

print("=== Chunk summary ===")
print(f"Total chunks    : {len(chunk_df):,}")
print(f"Unique filings  : {chunk_df['doc_id'].nunique():,}")
print(f"Unique companies: {chunk_df['company'].nunique():,}")
print(f"Avg sentences/chunk : {sent_chunk_map.groupby('chunk_id').size().mean():.1f}")
print(
    f"\u2705 sent_chunk_map saved \u2192 {OUT_DIR / 'sent_chunk_map.csv'}  ({len(sent_chunk_map):,} rows)"
)
display(chunk_df.head(8))


=== Chunk summary ===
Total chunks    : 97,595
Unique filings  : 17,560
Unique companies: 473
Avg sentences/chunk : 4.6
✅ sent_chunk_map saved → datasets/webapp_data/sent_chunk_map.csv  (452,390 rows)


,chunk_id,doc_id,company,quarter,year,chunk_text
0,c0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,presently the companys product line includes a...
1,c1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,credit card sales minimize accounts receivable...
2,c2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,under this method deferred tax assets and liab...
3,c3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,historically the advertising environment fluct...
4,c4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,the board established a february NUM NUM recor...
5,c5,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,Q1,2020,to date we have paid for any needed additions ...
6,c6,1-800-PetMeds_10-K_2020-05-26,1-800-PetMeds,Q2,2020,approximately NUM of all sales were generated ...
7,c7,1-800-PetMeds_10-K_2020-05-26,1-800-PetMeds,Q2,2020,due to covid- NUM consumer demand has increase...


## 2. Experiment 1: Base BERTopic

Establish a baseline using **all-MiniLM-L6-v2** sentence embeddings with default HDBSCAN
clustering. This gives us reference coherence and topic count values that each subsequent
experiment tries to improve.

### 4. Sentence Embeddings


In [8]:
EMBED_MODEL = "all-MiniLM-L6-v2"
print(f"Loading embedding model: {EMBED_MODEL} on device={DEVICE} ...")
embed_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)

BATCH_SIZE = 1024 if DEVICE == "cuda" else (256 if DEVICE == "mps" else 64)
print(f"Batch size : {BATCH_SIZE}  (device={DEVICE})")

_t0 = time.perf_counter()
embeddings = embed_model.encode(
    docs,
    show_progress_bar=True,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,  # unit-length → cosine ≡ dot, compatible with cuML UMAP
    convert_to_numpy=True,
)
# cuML UMAP expects float32 input
if embeddings.dtype != "float32":
    import numpy as np

    embeddings = embeddings.astype(np.float32)

_embed_sec = time.perf_counter() - _t0
print(f"✅ Embeddings shape : {embeddings.shape}  |  dtype: {embeddings.dtype}")
print(f"   Embedding time   : {_embed_sec:.1f}s")

Loading embedding model: all-MiniLM-L6-v2 on device=cuda ...
Batch size : 1024  (device=cuda)


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

✅ Embeddings shape : (97595, 384)  |  dtype: float32
   Embedding time   : 70.0s


In [9]:
# Pre-compute UMAP reduction ONCE — reused by every model fit and the sweep.
# cuML UMAP (GPU): ~10-20s on RTX 3090, no PCA init needed.
# CPU UMAP fallback (MPS machines): PCA warm-start for 3-5x faster convergence.
#
# MPS NOTE: PyTorch MPS is used for *embeddings* only. UMAP/HDBSCAN have no MPS
# backend and always run on CPU. Keep n_components=5 and n_neighbors=15:
#   • n_components=10 was unnecessarily high — standard BERTopic uses 5, and
#     HDBSCAN produces tighter clusters in lower-dimensional space.
#   • n_neighbors=50 on CPU with 50k+ chunks is very slow; 15 is the default
#     and gives better local structure for short financial text chunks.

_t1 = time.perf_counter()

if USE_CUML:
    print("Fitting UMAP on GPU (cuML) ...")
    _umap_base = GpuUMAP(
        n_components=5,
        n_neighbors=15,
        min_dist=0.0,
        metric="cosine",
        random_state=SEED,
    )
else:
    print("Fitting UMAP on CPU (PCA-initialised) ...")

    def _rescale(x):
        x = np.array(x, copy=True).astype(np.float32)
        x /= np.std(x[:, 0]) * 10_000
        return x

    _pca_init = _rescale(
        PCA(n_components=5, random_state=SEED).fit_transform(embeddings)
    )
    _umap_base = CpuUMAP(
        n_components=5,  # lowered from 10 — standard BERTopic default
        n_neighbors=15,  # lowered from 50 — much faster on CPU, better local structure
        min_dist=0.0,
        metric="cosine",
        init=_pca_init,
        low_memory=True,  # trades ~10% speed for lower RAM on large chunk sets
        random_state=SEED,
    )

umap_embeddings = _umap_base.fit_transform(embeddings)
_umap_sec = time.perf_counter() - _t1
backend = "GPU cuML" if USE_CUML else "CPU PCA-init"
print(
    f"✅ UMAP shape : {umap_embeddings.shape}  |  time: {_umap_sec:.1f}s  ({backend})"
)


class PassthroughUMAP(BaseEstimator):
    """Returns pre-computed UMAP embeddings — skips re-fitting for all downstream calls."""

    def __init__(self, precomputed):
        self.precomputed = precomputed

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return self.precomputed

    def fit_transform(self, X, y=None):
        return self.precomputed


print("✅ PassthroughUMAP ready.")


Fitting UMAP on GPU (cuML) ...
✅ UMAP shape : (97595, 5)  |  time: 1.3s  (GPU cuML)
✅ PassthroughUMAP ready.


In [10]:
# 2-D UMAP for visualize_topics() — fitted on a 5 000-point subsample
# of the original embeddings so topic centroids (384-dim) can be transformed.
# PassthroughUMAP only returns precomputed 5-D doc embeddings and cannot
# project new points (topic centroids), so we keep a separate viz reducer.
_rng = np.random.default_rng(SEED)
_sample_idx = _rng.choice(
    len(embeddings), size=min(5_000, len(embeddings)), replace=False
)
_viz_umap = CpuUMAP(n_components=2, n_neighbors=15, metric="cosine", random_state=SEED)
_viz_umap.fit(embeddings[_sample_idx])
print("✅ 2-D viz UMAP ready.")


✅ 2-D viz UMAP ready.


### 5. Base Model Fit


In [11]:
umap_model = PassthroughUMAP(umap_embeddings)

if USE_CUML:
    hdbscan_model = GpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        gen_min_span_tree=True,
        prediction_data=True,
    )
else:
    hdbscan_model = CpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        prediction_data=True,
    )

vectorizer = CountVectorizer(
    stop_words="english",
    max_df=0.85,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE,
)
ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    ctfidf_model=ctfidf,
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True,
)

_t3 = time.perf_counter()
topics, probs = topic_model.fit_transform(docs, embeddings)
_initial_sec = time.perf_counter() - _t3
print(f"✅ Initial model fit time: {_initial_sec:.1f}s")

# Reassign outliers to nearest topic by embedding similarity
# Must use original 384-dim embeddings (not umap_embeddings) to match topic_embeddings_ dimensionality
topics = topic_model.reduce_outliers(
    docs, topics, strategy="embeddings", embeddings=embeddings, threshold=0.0
)
topic_model.update_topics(docs, topics=topics)

n_topics = len(set(t for t in topics if t != -1))
n_outliers = sum(1 for t in topics if t == -1)
print(f"   Topics found : {n_topics}")
print(f"   Outliers     : {n_outliers:,} ({n_outliers / len(docs):.1%})")

# wb.log_metrics(
#     {
#         "initial_fit_time_sec": float(_initial_sec),
#         "initial_topics_found": int(n_topics),
#         "initial_outliers": int(n_outliers),
#         "initial_outlier_ratio": float(n_outliers / len(docs)),
#     }
# )
print("ℹ️ W&B metrics logging is disabled for initial model fit.")

2026-04-04 22:05:55,708 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-04 22:05:55,709 - BERTopic - Dimensionality - Completed ✓
2026-04-04 22:05:55,710 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-04 22:06:02,974 - BERTopic - Cluster - Completed ✓
2026-04-04 22:06:02,990 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-04 22:06:23,358 - BERTopic - Representation - Completed ✓


✅ Initial model fit time: 34.3s


2026-04-04 22:06:31,424 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


   Topics found : 269
   Outliers     : 0 (0.0%)
ℹ️ W&B metrics logging is disabled for initial model fit.


In [12]:
# -- Coherence helper (Gensim Cv) - used throughout the notebook ----------------
import os

_N_WORKERS = max(1, os.cpu_count() - 1)

# Pre-build tokenised corpus + dictionary once - reused by every coherence call
_tokenised = [d.lower().split() for d in docs]
_dictionary = Dictionary(_tokenised)


def compute_coherence(model, stage=None, extra_log=None):
    """Compute Cv coherence; W&B logging is temporarily disabled."""
    t0 = time.perf_counter()
    topic_words = [
        [w for w, _ in words if w and " " not in w]  # skip ngrams not in unigram corpus
        for tid, words in model.get_topics().items()
        if tid != -1
    ]

    if not topic_words:
        score = 0.0
    else:
        try:
            cm = CoherenceModel(
                topics=topic_words,
                texts=_tokenised,
                dictionary=_dictionary,
                coherence="c_v",
                processes=_N_WORKERS,
            )
            score = float(cm.get_coherence())
        except Exception:
            score = 0.0

    elapsed = time.perf_counter() - t0

    # if stage is not None and "wb" in globals():
    #     payload = extra_log.copy() if isinstance(extra_log, dict) else {}
    #     wb.log_coherence(
    #         stage=stage,
    #         score=score,
    #         elapsed_sec=elapsed,
    #         **payload,
    #     )

    return score, elapsed

In [13]:
# ── Topic Names via get_topic_info() ─────────────────────────────────────────
topic_info = topic_model.get_topic_info()
print("=== Topic Overview ===")
display(topic_info)

print("\n=== Topic Names ===")
for _, row in topic_info.iterrows():
    if row["Topic"] != -1:
        print(f"  Topic {row['Topic']:>3}: {row['Name']}")

=== Topic Overview ===


,Topic,Count,Name,Representation,Representative_Docs
0,0,2384,0_goodwill_impairment_carrying_reporting,"[goodwill, impairment, carrying, reporting, va...",[the first step requires the identification of...
1,1,3452,1_tax_income_rate_effective,"[tax, income, rate, effective, taxes, months, ...",[our foreign currency gains and losses primari...
2,2,1603,2_bitcoin_mining_miners_digital,"[bitcoin, mining, miners, digital, crypto, blo...",[with regard to the note receivable from roc d...
3,3,2041,3_asu_guidance_fasb_adoption,"[asu, guidance, fasb, adoption, standard, begi...",[the amendments in this guidance are effective...
4,4,1673,4_covid_pandemic_impact_employees,"[covid, pandemic, impact, employees, continue,...",[for example during the covid- NUM pandemic we...
...,...,...,...,...,...
264,264,107,264_absolute_dollars_aws_fulfillment,"[absolute, dollars, aws, fulfillment, comparab...",[the increase in international operating loss ...
265,265,190,265_currency_fiscal_constant_hardware,"[currency, fiscal, constant, hardware, fluctua...",[excluding the effect of foreign currency rate...
266,266,85,266_estimates_assumptions_actual_differ,"[estimates, assumptions, actual, differ, resul...",[estimates of fair value are based upon assump...
267,267,56,267_eksonr_ekso_exoskeleton_rehabilitation,"[eksonr, ekso, exoskeleton, rehabilitation, in...",[these benefits include a reduction in seconda...



=== Topic Names ===
  Topic   0: 0_goodwill_impairment_carrying_reporting
  Topic   1: 1_tax_income_rate_effective
  Topic   2: 2_bitcoin_mining_miners_digital
  Topic   3: 3_asu_guidance_fasb_adoption
  Topic   4: 4_covid_pandemic_impact_employees
  Topic   5: 5_nongaap_gaap_measures_operating
  Topic   6: 6_notes_convertible_conversion_note
  Topic   7: 7_privacy_data_gdpr_protection
  Topic   8: 8_recognized_contract_contracts_revenue
  Topic   9: 9_common_stock_class_shares
  Topic  10: 10_expenses_costs_development_administrative
  Topic  11: 11_employees_employee_talent_culture
  Topic  12: 12_internal_control_reporting_financial
  Topic  13: 13_credit_facility_revolving_loan
  Topic  14: 14_intellectual_property_rights_patents
  Topic  15: 15_awards_compensation_grant_stockbased
  Topic  16: 16_tax_deferred_assets_valuation
  Topic  17: 17_security_incidents_or_attacks
  Topic  18: 18_court_filed_complaint_motion
  Topic  19: 19_reinsurance_insurance_premium_catastrophe
  Topic

### 6. Visualisations — Base Model

#### Intertopic Distance Map

Each bubble represents one topic; bubble **size** is proportional to the number of
documents assigned; **distance** approximates semantic difference.

#### Term Score Bar Chart

Bar height = c-TF-IDF score — how distinctive a term is to its topic vs. all others.


In [14]:
import plotly.io as pio

pio.renderers.default = "colab"  # Setting explicitly to 'colab'

# ── Intertopic Distance Map (with topic colours) ───────────────────────────────
# Temporarily swap in the 2-D viz UMAP so BERTopic can project topic centroids.
_saved_umap = topic_model.umap_model
topic_model.umap_model = _viz_umap
fig_idm = topic_model.visualize_topics()
topic_model.umap_model = _saved_umap  # restore PassthroughUMAP
fig_idm.update_layout(
    title_text="Intertopic Distance Map — Initial Model",
    height=600,
)
fig_idm

In [15]:
import plotly.io as pio

pio.renderers.default = "colab"  # Setting explicitly to 'colab'

# ── Term Score Bar Charts ─────────────────────────────────────────────────────
n_show = min(n_topics, 9)  # cap at 9 for readability
fig_bc = topic_model.visualize_barchart(top_n_topics=n_show, n_words=8)
fig_bc.update_layout(
    title_text="Top Terms per Topic — c-TF-IDF Scores (Initial Model)",
    height=700,
)
fig_bc

### 7. Coherence Validation — Experiment 1

Hyperparameters are fixed to **Run 2** values (`min_cluster_size=50`, `min_samples=15`,
`min_df=5`, `ngram_range=(1,3)`). We compute a single Cv coherence score as our
Experiment 1 baseline.


In [ ]:
# Single coherence check for Run 2 params - no sweep loop needed.
cv_run2, _cv_sec = compute_coherence(
    topic_model,
    stage="run2",
    extra_log={
        "topics": int(n_topics),
        "min_cluster_size": int(MIN_CLUSTER_SIZE),
        "min_samples": int(MIN_SAMPLES),
    },
)

k_results = [{"mcs": MIN_CLUSTER_SIZE, "k": n_topics, "cv": cv_run2}]
best_k_row = k_results[0]

print(
    f"{'min_cluster_size':>18} {'min_samples':>12} {'k (topics)':>12} {'Coherence Cv':>14}"
)
print("-" * 62)
print(f"{MIN_CLUSTER_SIZE:>18} {MIN_SAMPLES:>12} {n_topics:>12} {cv_run2:>14.4f}")
print(f"\nCoherence computed in {_cv_sec:.1f}s")
print("ℹ️ W&B coherence logging is disabled.")

In [ ]:
# Simple coherence summary bar for the single Run-2 configuration
fig, ax = plt.subplots(figsize=(5, 3))
ax.barh(["Run 2"], [cv_run2], color="#2980b9", height=0.4)
ax.axvline(0.55, color="#27ae60", linestyle="--", lw=1.2, label="Good (0.55)")
ax.axvline(0.70, color="#e74c3c", linestyle="--", lw=1.2, label="Very Good (0.70)")
ax.set_xlim(0, max(1.0, cv_run2 + 0.1))
ax.set_xlabel("Coherence Cv", fontsize=11)
ax.set_title(
    f"Run 2 — Cv={cv_run2:.4f}  |  k={n_topics} topics", fontsize=12, fontweight="bold"
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 3. Experiment 2: Reducing Clusters

Building on the base model from Experiment 1, we evaluate three strategies to
consolidate fragmented or over-split topics into a more meaningful set.

| Strategy         | Method                                                | Effect                            |
| ---------------- | ----------------------------------------------------- | --------------------------------- |
| Merge Topics     | `merge_topics()` — joins highly similar topic pairs   | Removes near-duplicate topics     |
| Reduce Topics    | `reduce_topics(nr_topics=N)` — hierarchical reduction | Forces a fixed target count       |
| Min Cluster Size | Refit HDBSCAN with larger `min_cluster_size`          | Fewer, broader clusters at source |

The best-performing strategy is carried forward to Experiment 3.


### 2.1 Merge Topics

`merge_topics()` locates topic pairs whose c-TF-IDF vectors are highly similar (cosine ≥
threshold) and merges them in-place. This eliminates near-duplicate topics without a full
refit.


In [ ]:
# ── 2.1 Merge Topics ─────────────────────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Fresh model copy so Experiment 1 results are untouched
umap_e2m = PassthroughUMAP(umap_embeddings)
hdbscan_e2m = (
    GpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        gen_min_span_tree=True,
        prediction_data=True,
    )
    if USE_CUML
    else CpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        prediction_data=True,
    )
)
topic_model_e2m = BERTopic(
    umap_model=umap_e2m,
    hdbscan_model=hdbscan_e2m,
    vectorizer_model=CountVectorizer(
        stop_words="english", max_df=0.85, min_df=MIN_DF, ngram_range=NGRAM_RANGE
    ),
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    top_n_words=10,
    calculate_probabilities=False,
    verbose=False,
)
topics_e2m, _ = topic_model_e2m.fit_transform(docs, embeddings)
n_before_merge = len(set(topics_e2m)) - (1 if -1 in topics_e2m else 0)

# Find pairs above similarity threshold
MERGE_THRESHOLD = 0.80
topic_vecs = np.array(topic_model_e2m.topic_embeddings_[1:])  # skip outlier row
sim_mat = cos_sim(topic_vecs)
topic_ids_e2m = sorted(t for t in topic_model_e2m.get_topics() if t != -1)

pairs_to_merge, merged_set = [], set()
for _i in range(len(topic_ids_e2m)):
    for _j in range(_i + 1, len(topic_ids_e2m)):
        if sim_mat[_i, _j] >= MERGE_THRESHOLD:
            ti, tj = topic_ids_e2m[_i], topic_ids_e2m[_j]
            if ti not in merged_set and tj not in merged_set:
                pairs_to_merge.append([ti, tj])
                merged_set.update([ti, tj])

print(f"Topic pairs to merge (sim ≥ {MERGE_THRESHOLD}): {len(pairs_to_merge)}")
if pairs_to_merge:
    topic_model_e2m.merge_topics(docs, pairs_to_merge)

topics_e2m = topic_model_e2m.topics_
n_after_merge = len(set(topics_e2m)) - (1 if -1 in topics_e2m else 0)
cv_e2_merge, _ = compute_coherence(topic_model_e2m, stage="e2_merge")
print(f"Topics  : {n_before_merge} → {n_after_merge}  |  Cv: {cv_e2_merge:.4f}")


### 2.2 Reduce Topics

`reduce_topics(nr_topics=N)` hierarchically merges the most similar topics until the
count reaches `N`. Useful when you have a target granularity in mind.


In [ ]:
# ── 2.2 Reduce Topics ────────────────────────────────────────────────────────
TARGET_TOPICS = max(5, n_topics // 2)  # halve the Exp 1 topic count as target

umap_e2r = PassthroughUMAP(umap_embeddings)
hdbscan_e2r = (
    GpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        gen_min_span_tree=True,
        prediction_data=True,
    )
    if USE_CUML
    else CpuHDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        prediction_data=True,
    )
)
topic_model_e2r = BERTopic(
    umap_model=umap_e2r,
    hdbscan_model=hdbscan_e2r,
    vectorizer_model=CountVectorizer(
        stop_words="english", max_df=0.85, min_df=MIN_DF, ngram_range=NGRAM_RANGE
    ),
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    top_n_words=10,
    calculate_probabilities=False,
    verbose=False,
)
topics_e2r, _ = topic_model_e2r.fit_transform(docs, embeddings)
n_before_reduce = len(set(topics_e2r)) - (1 if -1 in topics_e2r else 0)
topic_model_e2r.reduce_topics(docs, nr_topics=TARGET_TOPICS)
topics_e2r = topic_model_e2r.topics_
n_after_reduce = len(set(topics_e2r)) - (1 if -1 in topics_e2r else 0)
cv_e2_reduce, _ = compute_coherence(topic_model_e2r, stage="e2_reduce")
print(
    f"Topics  : {n_before_reduce} → {n_after_reduce} (target={TARGET_TOPICS})  |  Cv: {cv_e2_reduce:.4f}"
)


### 2.3 Min Cluster Size

A larger `min_cluster_size` forces HDBSCAN to form fewer, broader clusters — a global
granularity control applied at fit time. We scan a short range and select the value with
the best Cv coherence score.


In [ ]:
# ── 2.3 Min Cluster Size sweep ───────────────────────────────────────────────
MCS_CANDIDATES = [30, 50, 75, 100, 150]
mcs_results = []

for _mcs in MCS_CANDIDATES:
    _ms = max(5, _mcs // 3)
    _tm = BERTopic(
        umap_model=PassthroughUMAP(umap_embeddings),
        hdbscan_model=(
            GpuHDBSCAN(
                min_cluster_size=_mcs,
                min_samples=_ms,
                metric="euclidean",
                gen_min_span_tree=True,
                prediction_data=True,
            )
            if USE_CUML
            else CpuHDBSCAN(
                min_cluster_size=_mcs,
                min_samples=_ms,
                metric="euclidean",
                prediction_data=True,
            )
        ),
        vectorizer_model=CountVectorizer(
            stop_words="english", max_df=0.85, min_df=MIN_DF, ngram_range=NGRAM_RANGE
        ),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        top_n_words=10,
        calculate_probabilities=False,
        verbose=False,
    )
    _topics, _ = _tm.fit_transform(docs, embeddings)
    _k = len(set(_topics)) - (1 if -1 in _topics else 0)
    _out_pct = 100 * sum(1 for t in _topics if t == -1) / len(_topics)
    _cv, _ = compute_coherence(_tm, stage=f"mcs_{_mcs}")
    mcs_results.append(
        {
            "mcs": _mcs,
            "k": _k,
            "outlier_pct": _out_pct,
            "cv": _cv,
            "model": _tm,
            "topics": _topics,
        }
    )
    print(f"mcs={_mcs:>4}  k={_k:>3}  outliers={_out_pct:.1f}%  Cv={_cv:.4f}")

mcs_df = pd.DataFrame(
    [{k: v for k, v in r.items() if k != "model"} for r in mcs_results]
)
best_mcs_idx = mcs_df["cv"].idxmax()
best_mcs_row = mcs_results[best_mcs_idx]
best_mcs = int(best_mcs_row["mcs"])
print(
    f"\nBest: mcs={best_mcs}  k={best_mcs_row['k']}  Cv={best_mcs_row['cv']:.4f}  → carried to Exp 3"
)


In [ ]:
# ── Experiment 2 Summary ─────────────────────────────────────────────────────
print("=" * 65)
print(f"{'Strategy':<30} {'Topics':>7} {'Cv Coherence':>14}")
print("-" * 65)
print(f"{'Exp 1 — Base (MiniLM)':<30} {n_topics:>7} {cv_run2:>14.4f}")
print(f"{'Exp 2.1 — Merge Topics':<30} {n_after_merge:>7} {cv_e2_merge:>14.4f}")
print(f"{'Exp 2.2 — Reduce Topics':<30} {n_after_reduce:>7} {cv_e2_reduce:>14.4f}")
print(
    f"{'Exp 2.3 — Min Cluster Size':<30} {best_mcs_row['k']:>7} {best_mcs_row['cv']:>14.4f}"
)
print("=" * 65)
print(f"\n→ Exp 2 winner: Min Cluster Size (mcs={best_mcs}) — carried forward to Exp 3")

# Bar chart comparison
import matplotlib.pyplot as plt

labels = [
    "Exp 1\nBase",
    "2.1 Merge\nTopics",
    "2.2 Reduce\nTopics",
    f"2.3 MCS\n(mcs={best_mcs})",
]
cv_vals = [cv_run2, cv_e2_merge, cv_e2_reduce, best_mcs_row["cv"]]
colors = ["#95a5a6", "#3498db", "#3498db", "#2ecc71"]
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(labels, cv_vals, color=colors, edgecolor="white", linewidth=0.8)
ax.axhline(0.55, color="#27ae60", linestyle="--", lw=1.2, label="Good (0.55)")
ax.axhline(0.70, color="#e74c3c", linestyle="--", lw=1.2, label="Very Good (0.70)")
for bar, val in zip(bars, cv_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{val:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("Cv Coherence")
ax.set_title("Experiment 2 — Cluster Reduction Strategies", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 4. Experiment 3: Use FinBERT Embeddings

Replaces the general-purpose **all-MiniLM-L6-v2** encoder with **ProsusAI/finbert** —
the same model used for sentiment analysis in this project. Domain-specific embeddings
capture financial language nuances that a general model may miss, potentially producing
tighter and more coherent topic clusters.

The approach extracts mean-pooled last-hidden-state vectors (the same pooling used in
FinBERT fine-tuning). The best `min_cluster_size` from Experiment 2 is reused.


In [ ]:
# ── Exp 3: Compute FinBERT embeddings ────────────────────────────────────────
from transformers import AutoTokenizer, AutoModel
import torch

FINBERT_MODEL_NAME = "ProsusAI/finbert"
print(f"Loading {FINBERT_MODEL_NAME} ...")
_fb_tok = AutoTokenizer.from_pretrained(FINBERT_MODEL_NAME)
_fb_model = AutoModel.from_pretrained(FINBERT_MODEL_NAME).to(DEVICE)
_fb_model.eval()


def encode_finbert(texts, batch_size=64, max_length=128):
    """Mean-pool last hidden state over non-padding tokens (L2-normalised)."""
    all_embs = []
    for _i in range(0, len(texts), batch_size):
        _batch = texts[_i : _i + batch_size]
        _enc = _fb_tok(
            _batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            _out = _fb_model(**_enc)
        _mask = _enc["attention_mask"].unsqueeze(-1).float()
        _emb = (_out.last_hidden_state * _mask).sum(1) / _mask.sum(1)
        if DEVICE in ("cuda", "mps"):
            _emb = _emb.cpu()
        all_embs.append(_emb.numpy())
        if (_i // batch_size) % 20 == 0:
            print(
                f"  {min(_i + batch_size, len(texts))}/{len(texts)} chunks encoded ..."
            )
    arr = np.vstack(all_embs).astype("float32")
    # L2-normalise for cosine compatibility
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / np.where(norms == 0, 1, norms)


_t3 = time.perf_counter()
embeddings_finbert = encode_finbert(docs)
print(
    f"\u2705 FinBERT embeddings: {embeddings_finbert.shape}  |  {time.perf_counter() - _t3:.1f}s"
)


In [ ]:
# ── Exp 3: UMAP reduction on FinBERT embeddings ─────────────────────────────
_t3u = time.perf_counter()
if USE_CUML:
    _umap3 = GpuUMAP(
        n_components=5, n_neighbors=15, metric="euclidean", random_state=SEED
    )
else:
    _umap3 = CpuUMAP(
        n_components=5,
        n_neighbors=15,
        metric="cosine",
        n_jobs=-1,
        random_state=SEED,
        init="pca" if len(docs) > 10_000 else "spectral",
    )
umap_embeddings_finbert = _umap3.fit_transform(embeddings_finbert)
if hasattr(umap_embeddings_finbert, "to_output"):
    umap_embeddings_finbert = umap_embeddings_finbert.to_output("numpy")
umap_embeddings_finbert = umap_embeddings_finbert.astype("float32")
print(
    f"\u2705 FinBERT UMAP: {umap_embeddings_finbert.shape}  |  {time.perf_counter() - _t3u:.1f}s"
)

# 2-D viz UMAP for FinBERT (used by visualize_topics in Exp 4)
_rng3 = np.random.default_rng(SEED)
_sub3 = _rng3.choice(
    len(embeddings_finbert), size=min(5_000, len(embeddings_finbert)), replace=False
)
_viz_umap_finbert = CpuUMAP(
    n_components=2, n_neighbors=15, metric="cosine", random_state=SEED
)
_viz_umap_finbert.fit(embeddings_finbert[_sub3])
print("\u2705 2-D viz UMAP (FinBERT) ready.")


In [ ]:
# ── Exp 3: Fit BERTopic with FinBERT embeddings ──────────────────────────────
topic_model_e3 = BERTopic(
    umap_model=PassthroughUMAP(umap_embeddings_finbert),
    hdbscan_model=(
        GpuHDBSCAN(
            min_cluster_size=best_mcs,
            min_samples=max(5, best_mcs // 3),
            metric="euclidean",
            gen_min_span_tree=True,
            prediction_data=True,
        )
        if USE_CUML
        else CpuHDBSCAN(
            min_cluster_size=best_mcs,
            min_samples=max(5, best_mcs // 3),
            metric="euclidean",
            prediction_data=True,
        )
    ),
    vectorizer_model=CountVectorizer(
        stop_words="english", max_df=0.85, min_df=MIN_DF, ngram_range=NGRAM_RANGE
    ),
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    top_n_words=10,
    calculate_probabilities=False,
    verbose=False,
)
topics_e3, _ = topic_model_e3.fit_transform(docs, embeddings_finbert)
n_topics_e3 = len(set(topics_e3)) - (1 if -1 in topics_e3 else 0)
n_outliers_e3 = sum(1 for t in topics_e3 if t == -1)
cv_e3, _ = compute_coherence(topic_model_e3, stage="e3_finbert")

print("=" * 60)
print(f"{'Model':<30} {'k':>5} {'Cv':>10}")
print("-" * 60)
print(f"{'Exp 1 — MiniLM base':<30} {n_topics:>5} {cv_run2:>10.4f}")
print(f"{'Exp 2 — MCS tuned':<30} {best_mcs_row['k']:>5} {best_mcs_row['cv']:>10.4f}")
print(f"{'Exp 3 — FinBERT embeds':<30} {n_topics_e3:>5} {cv_e3:>10.4f}")
print("=" * 60)
print(f"Outliers Exp 3: {n_outliers_e3:,} ({100 * n_outliers_e3 / len(docs):.1f}%)")


## 5. Experiment 4: KeyBERT Inspired Representation

Adds **KeyBERTInspired** as a representation model on top of Experiment 3's FinBERT-embedded
model. While c-TF-IDF generates topic keywords from term frequency alone, KeyBERTInspired
re-ranks candidates by cosine similarity to the document embeddings — producing labels that
are both statistically prominent and semantically central to each topic.

**This is the final, best model**, incorporating all improvements:

| Experiment | What changed                                 | Benefit                             |
| ---------- | -------------------------------------------- | ----------------------------------- |
| Exp 1      | all-MiniLM-L6-v2 embeddings, default HDBSCAN | baseline                            |
| Exp 2      | Tuned `min_cluster_size` via grid search     | better cluster count / coherence    |
| Exp 3      | Replaced embeddings with ProsusAI/finbert    | domain-aligned clustering           |
| Exp 4      | Added KeyBERTInspired representation         | richer, semantically precise labels |


In [ ]:
# ── Experiment 4: KeyBERT Inspired Representation ────────────────────────────
# Builds on Exp 2 (best_mcs) + Exp 3 (FinBERT embeddings).
print(f"Final model — min_cluster_size={best_mcs}  min_samples={max(5, best_mcs // 3)}")
print(f"             min_df={MIN_DF}  ngram_range={NGRAM_RANGE}")
print("Embedding : ProsusAI/finbert (Exp 3)")
print("Representation: KeyBERTInspired (Exp 4)")
print("-" * 50)

if USE_CUML:
    hdbscan_final = GpuHDBSCAN(
        min_cluster_size=best_mcs,
        min_samples=max(5, best_mcs // 3),
        metric="euclidean",
        gen_min_span_tree=True,
        prediction_data=True,
    )
else:
    hdbscan_final = CpuHDBSCAN(
        min_cluster_size=best_mcs,
        min_samples=max(5, best_mcs // 3),
        metric="euclidean",
        prediction_data=True,
    )

vectorizer_final = CountVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE,
)
ctfidf_final = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = KeyBERTInspired()

# PassthroughUMAP uses the pre-computed FinBERT UMAP embeddings (from Exp 3).
# embed_model (all-MiniLM) is kept so KeyBERTInspired can score keyword candidates.
topic_model_final = BERTopic(
    embedding_model=embed_model,
    umap_model=PassthroughUMAP(umap_embeddings_finbert),
    hdbscan_model=hdbscan_final,
    vectorizer_model=vectorizer_final,
    ctfidf_model=ctfidf_final,
    representation_model=representation_model,
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True,
)

_t4 = time.perf_counter()
topics_final, probs_final = topic_model_final.fit_transform(docs, embeddings_finbert)
_final_sec = time.perf_counter() - _t4
print(f"\u2705 Final model fit time: {_final_sec:.1f}s")

# Reassign outliers to nearest topic by FinBERT embedding similarity
topics_final = topic_model_final.reduce_outliers(
    docs,
    topics_final,
    strategy="embeddings",
    embeddings=embeddings_finbert,
    threshold=0.0,
)
topic_model_final.update_topics(docs, topics=topics_final)

n_topics_final = len(set(t for t in topics_final if t != -1))
n_outliers_final = sum(1 for t in topics_final if t == -1)
cv_final, _ = compute_coherence(topic_model_final, stage="final_keybert")
print(f"   Final Topics  : {n_topics_final}")
print(f"   Final Outliers: {n_outliers_final:,} ({n_outliers_final / len(docs):.1%})")
print(f"   Cv Coherence  : {cv_final:.4f}")


In [ ]:
# ── Final topic names via get_topic_info() → Name column ─────────────────────
topic_info_final = topic_model_final.get_topic_info()
print("=== Final Topic Overview (KeyBERT-enhanced) ===")
display(topic_info_final)

print("\n=== Final Topic Names ===")
for _, row in topic_info_final.iterrows():
    if row["Topic"] != -1:
        print(f"  Topic {row['Topic']:>3}: {row['Name']}")

In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook"  # Fix rendering in VSCode / Google Colab

# ── Final Intertopic Distance Map — uses FinBERT 2-D viz UMAP ────────────────
_saved_umap_final = topic_model_final.umap_model
topic_model_final.umap_model = _viz_umap_finbert
fig_final_idm = topic_model_final.visualize_topics()
topic_model_final.umap_model = _saved_umap_final
fig_final_idm.update_layout(
    title_text="Experiment 4 — Final Model: Intertopic Distance Map",
    height=600,
)
fig_final_idm


In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook"

# ── Final Term Score Bar Charts ───────────────────────────────────────────────
n_show_final = min(n_topics_final, 9)
fig_final_bc = topic_model_final.visualize_barchart(
    top_n_topics=n_show_final, n_words=8
)
fig_final_bc.update_layout(
    title_text="Experiment 4 — Final Model: Top Terms per Topic (KeyBERT Representation)",
    height=700,
)
fig_final_bc

## 6. Coherence Assessment

**Cv coherence** measures how semantically similar the top words within each topic are —
higher is better.

| Cv range  | Interpretation |
| --------- | -------------- |
| < 0.45    | Poor           |
| 0.45–0.55 | Marginal       |
| 0.55–0.70 | Good           |
| > 0.70    | Very good      |


In [ ]:
cv_initial, cv_initial_sec = compute_coherence(
    topic_model,
    stage="initial",
    extra_log={"topics": int(n_topics)},
)
cv_final, cv_final_sec = compute_coherence(
    topic_model_final,
    stage="final",
    extra_log={"topics": int(n_topics_final)},
)

# wb.log_metrics(
#     {
#         "coherence/improvement": float(cv_final - cv_initial),
#         "coherence/initial_time_sec": float(cv_initial_sec),
#         "coherence/final_time_sec": float(cv_final_sec),
#     }
# )

print("=" * 55)
print("COHERENCE ASSESSMENT")
print("=" * 55)
print(f"\n  Initial model Cv : {cv_initial:.4f}")
print(f"  Final model Cv   : {cv_final:.4f}")
print(f"  Improvement      : {cv_final - cv_initial:+.4f}")
print("ℹ️ W&B coherence logging is disabled.")

print()
for score, label in [(cv_initial, "Initial model"), (cv_final, "Final model")]:
    if score >= 0.70:
        msg = "VERY GOOD (>=0.70)"
    elif score >= 0.55:
        msg = "GOOD (0.55-0.70)"
    elif score >= 0.45:
        msg = "MARGINAL (0.45-0.55) - consider tuning"
    else:
        msg = "POOR (<0.45)"
    print(f"  {label:<16}: Cv = {score:.4f}  =>  {msg}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(
    ["Initial", "Final"],
    [cv_initial, cv_final],
    color=["#7fb3d3", "#2980b9"],
    edgecolor="white",
    width=0.4,
)
for bar, val in zip(bars, [cv_initial, cv_final]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.003,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )
ax.set_ylabel("Coherence Cv")
ax.set_title("Initial vs Final Model - Coherence", fontweight="bold")
ax.set_ylim(0, max(cv_initial, cv_final) * 1.2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Topic Diversity

**Topic Diversity** = unique words across all topics ÷ total word slots across all topics.

A high score (close to 1.0) means topics use largely distinct vocabulary — good separation.
A low score means topics share many words — possible redundancy or over-splitting.


In [ ]:
# ── Topic Diversity for final model ──────────────────────────────────────────
all_topic_words = []
for tid, words in topic_model_final.get_topics().items():
    if tid != -1:
        all_topic_words.extend([w for w, _ in words if w])

unique_words = set(all_topic_words)
topic_diversity = len(unique_words) / len(all_topic_words) if all_topic_words else 0.0

print("=" * 55)
print("TOPIC DIVERSITY — Final Model")
print("=" * 55)
print(f"\n  Total word slots : {len(all_topic_words)}")
print(f"  Unique words     : {len(unique_words)}")
print(f"  Topic Diversity  : {topic_diversity:.4f}  ({topic_diversity:.1%})")

if topic_diversity >= 0.70:
    div_msg = "HIGH — topics are well-differentiated"
elif topic_diversity >= 0.50:
    div_msg = "MODERATE — some vocabulary overlap between topics"
else:
    div_msg = "LOW — topics share much vocabulary; consider increasing k"
print(f"\n  Interpretation: {div_msg}")

# Visualise word frequency across topics
from collections import Counter

word_freq = Counter(all_topic_words)
top_shared = [(w, c) for w, c in word_freq.most_common(20) if c > 1]

if top_shared:
    words_s, counts_s = zip(*top_shared)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    colors = ["#e74c3c" if c >= 3 else "#e67e22" for c in counts_s]
    ax.barh(words_s, counts_s, color=colors, edgecolor="white")
    ax.invert_yaxis()
    ax.set_xlabel("Number of topics containing this word")
    ax.set_title(
        f"Words Shared Across Topics (Diversity = {topic_diversity:.1%})",
        fontweight="bold",
    )
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\n(Red bars = word appears in 3+ topics — high overlap indicator)")

## 8. Outlier Experimentation

Run this section if the final model has too many outliers (`Topic = -1`).  
Grid-searches `min_cluster_size` values and tries BERTopic's built-in
`reduce_outliers()` strategies to find the best balance of topic count vs outlier rate.


In [ ]:
# ── 11.1 Current outlier rate ────────────────────────────────────────────────
n_outliers_now = sum(1 for t in topics_final if t == -1)
n_total_now = len(topics_final)
print(
    f"Current outliers : {n_outliers_now:,} / {n_total_now:,} "
    f"({100 * n_outliers_now / n_total_now:.1f}%)"
)

# ── 11.2 Grid-search min_cluster_size ────────────────────────────────────────
# Lower values → more topics, fewer outliers; higher → fewer, broader topics.
mcs_candidates = [20, 30, 40, 50, 75, 100]
exp_results = []

print("\nRunning grid search ...")
for mcs in mcs_candidates:
    ms_exp = max(5, mcs // 5)
    if USE_CUML:
        hdb_exp = GpuHDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms_exp,
            metric="euclidean",
            prediction_data=True,
        )
    else:
        hdb_exp = CpuHDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms_exp,
            metric="euclidean",
            prediction_data=True,
        )

    model_exp = BERTopic(
        embedding_model=embed_model,
        umap_model=PassthroughUMAP(umap_embeddings),
        hdbscan_model=hdb_exp,
        vectorizer_model=CountVectorizer(
            stop_words="english", max_df=0.95, min_df=MIN_DF, ngram_range=NGRAM_RANGE
        ),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        calculate_probabilities=False,
        verbose=False,
    )
    topics_exp, _ = model_exp.fit_transform(docs, embeddings)
    # Reduce outliers with embedding similarity (threshold=0.1 keeps some as -1)
    topics_red = model_exp.reduce_outliers(
        docs, topics_exp, strategy="embeddings", embeddings=embeddings, threshold=0.1
    )
    n_out = sum(1 for t in topics_red if t == -1)
    n_top = len(set(t for t in topics_red if t != -1))
    exp_results.append(
        {
            "mcs": mcs,
            "n_topics": n_top,
            "n_outliers": n_out,
            "outlier_pct": 100 * n_out / n_total_now,
        }
    )
    print(
        f"  mcs={mcs:>3} | {n_top:>3} topics | "
        f"{n_out:>6,} outliers ({100 * n_out / n_total_now:.1f}%)"
    )

# ── 11.3 Visualise trade-off ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
mcs_vals = [r["mcs"] for r in exp_results]
ax1.plot(mcs_vals, [r["outlier_pct"] for r in exp_results], "o-", color="#e74c3c")
ax1.axhline(y=5, color="gray", linestyle="--", label="5% threshold")
ax1.set_xlabel("min_cluster_size")
ax1.set_ylabel("Outlier %")
ax1.set_title("Outlier Rate vs min_cluster_size")
ax1.legend()
ax2.plot(mcs_vals, [r["n_topics"] for r in exp_results], "s-", color="#3498db")
ax2.set_xlabel("min_cluster_size")
ax2.set_ylabel("Number of Topics")
ax2.set_title("Topic Count vs min_cluster_size")
plt.tight_layout()
plt.show()

# ── 11.4 Recommendation ───────────────────────────────────────────────────────
best_exp = min(exp_results, key=lambda r: r["outlier_pct"])
print(
    f"\nRecommended: min_cluster_size={best_exp['mcs']} "
    f"({best_exp['n_topics']} topics, {best_exp['outlier_pct']:.1f}% outliers)"
)
print("Update MIN_CLUSTER_SIZE in run2_config cell and re-run Sections 5 & 8.")


## 9. Topic Labeling

Inspect auto-generated topic keywords and assign human-readable labels.  
Edit `MANUAL_TOPIC_LABELS` with your chosen names, then re-run the cell.  
Common MD&A topics: _Revenue Growth, Supply Chain, Legal Risk, Liquidity,  
R&D Investment, Macro Outlook, Workforce & Talent, Cybersecurity & Privacy,  
AI & Competition, Shareholder Returns_


In [ ]:
# ── 12.1 Print auto-generated topic names + keywords ─────────────────────────
topic_info_final = topic_model_final.get_topic_info()
print("=== Auto-generated topic names + top 8 keywords ===")
print(f"{'Topic ID':<10} {'Auto Name':<55} Keywords")
print("-" * 110)
for _, row in topic_info_final.iterrows():
    tid = int(row["Topic"])
    if tid == -1:
        continue
    kws = ", ".join(w for w, _ in topic_model_final.get_topic(tid)[:8])
    print(f"{tid:<10} {row['Name']:<55} {kws}")

# ── 12.2 Manual label dict — EDIT THIS ───────────────────────────────────────
# Keys = topic IDs (int). Fill in after inspecting keywords above.
MANUAL_TOPIC_LABELS = {
    # 0:  'Revenue Growth',
    # 1:  'Supply Chain',
    # 2:  'Legal Risk',
    # 3:  'Liquidity & Capital',
    # 4:  'R&D Investment',
    # 5:  'Macro Outlook',
    # 6:  'Workforce & Talent',
    # 7:  'Cybersecurity & Privacy',
    # 8:  'AI & Competition',
    # 9:  'Shareholder Returns',
}

# ── 12.3 Build final label map (fallback to auto-name for unlabelled topics) ──
topic_labels: dict[int, str] = {}
for _, row in topic_info_final.iterrows():
    tid = int(row["Topic"])
    topic_labels[tid] = MANUAL_TOPIC_LABELS.get(tid, row["Name"])

topic_labels[-1] = "Outlier"

print("\n=== Final label map ===")
for tid, lbl in sorted(topic_labels.items()):
    count = topic_info_final.loc[topic_info_final["Topic"] == tid, "Count"].values
    cnt_str = f"{count[0]:,}" if len(count) else "—"
    print(f"  {tid:>4}: {lbl:<45}  ({cnt_str} chunks)")


## 10. Map Topics → Chunks & Export

Attaches BERTopic topic assignments to each chunk, then computes filing-level
topic weight distributions.

Outputs:

- **topic_chunk_df** — chunk_df + `topic_id` + `topic_label` (1st dataframe)
- **topic_dist_df** — filing-level topic weight distribution (2nd dataframe)
- `datasets/webapp_data/topic_chunk_df.csv`
- `datasets/webapp_data/topic_dist_df.csv`
- `models/bertopic_final/` — saved BERTopic model (SafeTensors format)


In [ ]:
from pathlib import Path
import pandas as pd

# ── 13.1 Attach topic assignments to chunk_df (Output 1) ─────────────────────
topic_chunk_df = chunk_df.copy()
topic_chunk_df["topic_id"] = topics_final
topic_chunk_df["topic_label"] = topic_chunk_df["topic_id"].map(topic_labels)

print("=== Output 1: topic_chunk_df (chunk + topic_label) ===")
display(
    topic_chunk_df[
        ["chunk_id", "doc_id", "company", "quarter", "topic_label", "chunk_text"]
    ].head(8)
)

# ── 13.2 Build topic_dist_df (Output 2) ───────────────────────────────────────
# Count chunks per (doc_id, topic), then normalise → weight (same concept as LDA θ).
topic_dist_df = (
    topic_chunk_df.groupby(["doc_id", "company", "quarter", "topic_label"])
    .size()
    .reset_index(name="chunk_count")
)
topic_dist_df["weight"] = topic_dist_df["chunk_count"] / topic_dist_df.groupby(
    "doc_id"
)["chunk_count"].transform("sum")

print("\n=== Output 2: topic_dist_df (filing-level topic distribution) ===")
display(topic_dist_df.head(12))

# ── 13.3 Save outputs ─────────────────────────────────────────────────────────
OUT_DIR = Path("datasets/webapp_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

topic_chunk_df.to_csv(OUT_DIR / "topic_chunk_df.csv", index=False)
topic_dist_df.to_csv(OUT_DIR / "topic_dist_df.csv", index=False)

print(
    f"\n\u2705 topic_chunk_df \u2192 {OUT_DIR / 'topic_chunk_df.csv'} ({len(topic_chunk_df):,} rows)"
)
print(
    f"\u2705 topic_dist_df  \u2192 {OUT_DIR / 'topic_dist_df.csv'} ({len(topic_dist_df):,} rows)"
)

# ── 13.4 Save BERTopic model ──────────────────────────────────────────────────
MODEL_DIR = Path("models/bertopic_final")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
topic_model_final.save(
    str(MODEL_DIR),
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=EMBED_MODEL,
)
print(f"\u2705 BERTopic model \u2192 {MODEL_DIR}")


## 11. Build final_df: FinBERT × BERTopic

Join sentence-level FinBERT scores with chunk-level BERTopic topics using
`sent_chunk_map` as the bridge.

```
finbert_inferenced.csv          (sentence_id, sentence, label, score, …)
      ↓  merge on sentence_id
sent_chunk_map                  (sentence_id → chunk_id)
      ↓  merge on chunk_id
topic_chunk_df                  (chunk_id → topic_label)
      ↓  merge on (doc_id, topic_label)
topic_dist_df                   (topic_label → weight per filing)
      ↓
final_df: doc_id | company | quarter | year | topic_label | topic_weight
          | sentence | score | label
```


In [ ]:
from pathlib import Path
import pandas as pd

FINBERT_PATH = Path("webapp/finbert_inferenced.csv")

if FINBERT_PATH.exists():
    sentiment_df = pd.read_csv(FINBERT_PATH)
    print(
        f"\u2705 Loaded FinBERT output: {len(sentiment_df):,} rows from {FINBERT_PATH}"
    )
else:
    # ── Sample data — replace with real FinBERT output when available ─────────
    sentiment_df = pd.DataFrame(
        {
            "doc_id": [1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 3.0, 3.0],
            "company": ["NVDA", "NVDA", "NVDA", "NVDA", "AAPL", "AAPL", "MSFT", "MSFT"],
            "quarter": ["Q4 2023"] * 8,
            "sentence": [
                "Data center revenue surpassed $18B, a 400% YoY increase.",
                "GPU demand accelerates across hyperscalers.",
                "Supply chain disruptions remain an ongoing operational risk.",
                "Geopolitical tensions may affect component availability.",
                "iPhone revenue declined 3% amid macro headwinds.",
                "Services segment achieved record revenue of $21B.",
                "Azure cloud revenue grew 29% driven by AI workloads.",
                "Supply chain normalisation has reduced our hardware costs.",
            ],
            "label": [
                "positive",
                "positive",
                "negative",
                "negative",
                "negative",
                "positive",
                "positive",
                "positive",
            ],
            "pos_proba": [0.92, 0.88, 0.05, 0.07, 0.08, 0.91, 0.89, 0.85],
            "neg_proba": [0.04, 0.06, 0.89, 0.86, 0.87, 0.04, 0.06, 0.09],
            "neutral_proba": [0.04, 0.06, 0.06, 0.07, 0.05, 0.05, 0.05, 0.06],
            "sentiment_score": [0.88, 0.82, -0.84, -0.79, -0.79, 0.87, 0.83, 0.76],
        }
    )
    FINBERT_PATH.parent.mkdir(parents=True, exist_ok=True)
    sentiment_df.to_csv(FINBERT_PATH, index=False)
    print(
        f"\u26a0\ufe0f  FinBERT output not found — sample data written to {FINBERT_PATH}"
    )

# sentence_id must match the row order assigned in Section 3 preprocessing
sentiment_df["sentence_id"] = sentiment_df.index

# Step 1: sentence → chunk_id  (via sent_chunk_map built in Section 3)
step1 = sentiment_df.merge(sent_chunk_map, on="sentence_id", how="left")

# Step 2: chunk_id → topic_label  (via topic_chunk_df built in Section 13)
step2 = step1.merge(
    topic_chunk_df[["chunk_id", "topic_label"]],
    on="chunk_id",
    how="left",
)

# Step 3: (doc_id, topic_label) → topic_weight  (via topic_dist_df)
step3 = step2.merge(
    topic_dist_df[["doc_id", "topic_label", "weight"]].rename(
        columns={"weight": "topic_weight"}
    ),
    on=["doc_id", "topic_label"],
    how="left",
)

# Extract year from quarter string (e.g. "Q4 2023" → 2023) if not present
if "year" not in step3.columns:
    step3["year"] = step3["quarter"].str.extract(r"(\d{4})").astype("Int64")

final_df = step3[
    [
        "doc_id",
        "company",
        "quarter",
        "year",
        "topic_label",
        "topic_weight",
        "sentence",
        "sentiment_score",
        "label",
    ]
].rename(columns={"sentiment_score": "score"})

print("\n=== final_df ===")
display(final_df.head(12))
print(f"\nShape : {final_df.shape}")
print(f"Topics: {final_df['topic_label'].nunique()} unique")

OUT_DIR = Path("datasets/webapp_data")
final_df.to_csv(OUT_DIR / "final_df.csv", index=False)
print(f"\u2705 final_df saved \u2192 {OUT_DIR / 'final_df.csv'}")


## 12. Random Search for Outlier Reduction (BERTopic)

Use this as an alternative to full grid search when outliers are high.

What this does:

- Randomly samples BERTopic/HDBSCAN/vectorizer settings
- Optionally applies `reduce_outliers(..., strategy='embeddings')`
- Ranks trials by a score that prioritizes low outlier rate while discouraging extreme topic counts

Tip:

- Start with `n_iter=10` for a quick pass
- Increase to `20-40` for a more stable search


In [ ]:
# Random-search utility for BERTopic outlier tuning
from sklearn.model_selection import ParameterSampler


def run_bertopic_random_search(
    n_iter=12,
    random_state=42,
    apply_reduce_outliers=True,
    target_topics=(6, 20),
):
    required_names = [
        "docs",
        "embeddings",
        "embed_model",
        "umap_embeddings",
        "USE_CUML",
        "GpuHDBSCAN",
        "CpuHDBSCAN",
        "PassthroughUMAP",
        "ClassTfidfTransformer",
        "BERTopic",
    ]
    missing = [n for n in required_names if n not in globals()]
    if missing:
        raise RuntimeError(
            "Missing required objects in current kernel: "
            + ", ".join(missing)
            + ". Re-run earlier BERTopic setup/training sections first."
        )

    param_space = {
        "min_cluster_size": [15, 20, 30, 40, 50, 75, 100],
        "min_samples_ratio": [0.15, 0.20, 0.25, 0.30],
        "min_df": [3, 5, 8, 12],
        "ngram_upper": [1, 2],
        "outlier_threshold": [0.0, 0.05, 0.10, 0.15],
    }

    sampler = list(
        ParameterSampler(param_space, n_iter=n_iter, random_state=random_state)
    )
    n_total = len(docs)
    results = []

    print(f"Running random search with {len(sampler)} trials...")

    for i, cfg in enumerate(sampler, 1):
        mcs = int(cfg["min_cluster_size"])
        min_samples = max(5, int(round(mcs * float(cfg["min_samples_ratio"]))))

        if USE_CUML:
            hdb_exp = GpuHDBSCAN(
                min_cluster_size=mcs,
                min_samples=min_samples,
                metric="euclidean",
                prediction_data=True,
            )
        else:
            hdb_exp = CpuHDBSCAN(
                min_cluster_size=mcs,
                min_samples=min_samples,
                metric="euclidean",
                prediction_data=True,
            )

        vec_exp = CountVectorizer(
            stop_words="english",
            max_df=0.95,
            min_df=int(cfg["min_df"]),
            ngram_range=(1, int(cfg["ngram_upper"])),
        )

        model_exp = BERTopic(
            embedding_model=embed_model,
            umap_model=PassthroughUMAP(umap_embeddings),
            hdbscan_model=hdb_exp,
            vectorizer_model=vec_exp,
            ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
            calculate_probabilities=False,
            verbose=False,
        )

        topics_raw, _ = model_exp.fit_transform(docs, embeddings)

        if apply_reduce_outliers:
            topics_use = model_exp.reduce_outliers(
                docs,
                topics_raw,
                strategy="embeddings",
                embeddings=embeddings,
                threshold=float(cfg["outlier_threshold"]),
            )
            model_exp.update_topics(docs, topics=topics_use)
        else:
            topics_use = topics_raw

        n_outliers = int(sum(1 for t in topics_use if t == -1))
        outlier_pct = 100 * n_outliers / n_total
        n_topics = int(len(set(t for t in topics_use if t != -1)))

        lo, hi = target_topics
        topic_penalty = 0.0 if lo <= n_topics <= hi else abs(n_topics - (lo + hi) / 2)

        # Higher score is better: low outlier % and reasonable topic count
        score = -(outlier_pct + 1.25 * topic_penalty)

        results.append(
            {
                "trial": i,
                "score": score,
                "outlier_pct": outlier_pct,
                "n_topics": n_topics,
                "n_outliers": n_outliers,
                "min_cluster_size": mcs,
                "min_samples": min_samples,
                "min_df": int(cfg["min_df"]),
                "ngram_upper": int(cfg["ngram_upper"]),
                "outlier_threshold": float(cfg["outlier_threshold"]),
                "model": model_exp,
                "topics": topics_use,
            }
        )

        print(
            f"[{i:>2}/{len(sampler)}] mcs={mcs:>3} ms={min_samples:>2} "
            f"min_df={int(cfg['min_df']):>2} ngram<= {int(cfg['ngram_upper'])} "
            f"thr={float(cfg['outlier_threshold']):.2f} | "
            f"topics={n_topics:>3} outliers={outlier_pct:>5.1f}% score={score:>7.2f}"
        )

    res_df = (
        pd.DataFrame(results)
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )

    cols = [
        "trial",
        "score",
        "outlier_pct",
        "n_topics",
        "n_outliers",
        "min_cluster_size",
        "min_samples",
        "min_df",
        "ngram_upper",
        "outlier_threshold",
    ]

    print("\nTop random-search trials:")
    display(res_df[cols].head(10))

    return res_df


In [ ]:
# Example run + pick best model
RUN_RANDOM_SEARCH = False  # set True to execute

if RUN_RANDOM_SEARCH:
    rs_results = run_bertopic_random_search(
        n_iter=12,
        random_state=42,
        apply_reduce_outliers=True,
        target_topics=(6, 20),
    )

    best = rs_results.iloc[0]
    topic_model_random_best = best["model"]
    topics_random_best = best["topics"]

    print("\nBest random-search configuration:")
    print(
        best[
            [
                "score",
                "outlier_pct",
                "n_topics",
                "n_outliers",
                "min_cluster_size",
                "min_samples",
                "min_df",
                "ngram_upper",
                "outlier_threshold",
            ]
        ]
    )

    # Optional: compare to current final model in one line
    curr_outlier_pct = 100 * sum(1 for t in topics_final if t == -1) / len(topics_final)
    new_outlier_pct = float(best["outlier_pct"])
    print(f"\nCurrent final outlier%: {curr_outlier_pct:.2f}%")
    print(f"Random-search best outlier%: {new_outlier_pct:.2f}%")
else:
    print("Set RUN_RANDOM_SEARCH = True to run random search.")